<a href="https://colab.research.google.com/github/gayakarapetyan/PythonCourse/blob/main/SimpleDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Simple demonstration



---

What can we do with Python ?

In [ ]:
#import libraries
import pandas as pd
import numpy as np

import plotly.graph_objects as go

import wetterdienst
#print(wetterdienst.__version__)
from wetterdienst.provider.dwd.observation import DwdObservationRequest
from wetterdienst import Settings

from sklearn.ensemble import RandomForestRegressor

In [ ]:
request = DwdObservationRequest(
    parameters=[("daily", "climate_summary", "temperature_air_mean_2m")],
    start_date="2025-02-01",
    end_date="2025-05-07",
)

values = request.filter_by_name("Magdeburg").values.all().df

df_temp = values.filter(values["parameter"] == "temperature_air_mean_2m")[["date", "value"]].to_pandas()
df_temp.columns = ["date", "temp_c"]
df_temp = df_temp.dropna().reset_index(drop=True)

print(f"Got {len(df_temp)} days of temperature data")
print(df_temp.tail(5))

Got 96 days of temperature data
                        date  temp_c
91 2025-05-03 00:00:00+00:00    13.1
92 2025-05-04 00:00:00+00:00    10.1
93 2025-05-05 00:00:00+00:00     9.5
94 2025-05-06 00:00:00+00:00    10.4
95 2025-05-07 00:00:00+00:00    12.0


In [ ]:
df = df_temp.copy()
df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
df = df.sort_values("date").reset_index(drop=True)

df["day_of_year"] = df["date"].dt.dayofyear
df["day_of_week"] = df["date"].dt.dayofweek
df["lag_1"] = df["temp_c"].shift(1)
df["lag_7"] = df["temp_c"].shift(7)

df = df.dropna().reset_index(drop=True)

In [ ]:
features = ["day_of_year", "day_of_week", "lag_1", "lag_7"]
X = df[features]
y = df["temp_c"]

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X, y)

print(f"R² score: {model.score(X, y):.2f}")

R² score: 0.97


In [ ]:
forecast_rows = []
last_known = df.copy()

for i in range(1, 8):
    next_date = last_known["date"].iloc[-1] + pd.Timedelta(days=1)
    lag_1 = last_known["temp_c"].iloc[-1]
    lag_7 = last_known["temp_c"].iloc[-7]
    day_of_year = next_date.dayofyear
    day_of_week = next_date.dayofweek

    pred = model.predict(pd.DataFrame([[day_of_year, day_of_week, lag_1, lag_7]], columns=features))[0]

    forecast_rows.append({"date": next_date, "temp_c": pred})
    new_row = pd.DataFrame([{"date": next_date, "temp_c": pred,
                              "day_of_year": day_of_year, "day_of_week": day_of_week,
                              "lag_1": lag_1, "lag_7": lag_7}])
    last_known = pd.concat([last_known, new_row], ignore_index=True)

df_forecast = pd.DataFrame(forecast_rows)
print(df_forecast)

        date   temp_c
0 2025-05-08  13.2130
1 2025-05-09  13.2600
2 2025-05-10  12.6645
3 2025-05-11  12.0885
4 2025-05-12  12.9165
5 2025-05-13  13.3710
6 2025-05-14  13.7405


In [18]:
#Now we can plot the temperature forecast for next 7 days

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["date"], y=df["temp_c"],
    mode="lines", name="Actual temperature",
    line=dict(color="#1565C0", width=2)
))

fig.add_trace(go.Scatter(
    x=df_forecast["date"], y=df_forecast["temp_c"],
    mode="lines+markers", name="7-day forecast",
    line=dict(color="orange", width=2, dash="dash"),
    marker=dict(size=8, color="orange")
))

fig.update_layout(
    title="Magdeburg temperature — actual + 7-day ML forecast",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", linewidth=1.5,
               ticks="outside", tickcolor="black"),
    yaxis=dict(showgrid=False, linecolor="black", linewidth=1.5,
               ticks="outside", tickcolor="black",
               title="Temperature (°C)"),
    font=dict(color="black"),
    title_font=dict(size=16),
    legend=dict(bgcolor="white", bordercolor="black", borderwidth=1)
)

fig.show()